# Daily Practice — 2026-10-20: Statistics — How Confident Are We, and Has the World Shifted?

**Dataset:** [Wine recognition dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) (`sklearn.datasets.load_wine`) — 178 real chemical analyses of wines grown in the same region of Italy, from three different cultivars (grape varieties), 13 continuous features (alcohol, malic acid, color intensity, proline, etc.). This is real laboratory measurement data, not a synthetic toy.

**Scenario:** You've moved from SQA into ML quality engineering. A teammate trained a classifier that predicts the wine cultivar from its chemical profile and reports "98% test accuracy" in Slack. Two questions an SQA mindset should immediately ask:

1. **How confident should we be in that single number?** A test set of ~54 rows can make a mediocre model look great (or a great model look mediocre) by luck of the draw. You need an interval, not a point estimate.
2. **Will that number still be true in production?** If the sensors, suppliers, or harvest conditions feeding the model shift over time, the *input* distribution drifts away from what the model was trained on, and the reported accuracy silently stops applying — long before anyone notices a drop in a dashboard.

**What you should produce:**
1. `bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot, alpha, random_state)` — a percentile-bootstrap confidence interval for *any* sklearn-style metric function, returning `(lower, point_estimate, upper)`.
2. Confidence intervals (not just point estimates) for test-set **accuracy** and **macro-F1** of a trained classifier.
3. `ks_drift_report(reference_df, current_df, alpha)` — a per-feature two-sample Kolmogorov–Smirnov drift check between a reference distribution and a "current" (production) distribution, returning a DataFrame with the KS statistic, p-value, and a Bonferroni-corrected `flagged` column (correcting for running one test per feature instead of just one).
4. A simulated "production batch" where you deliberately shift 2 of the 13 features (standing in for a sensor recalibration or a new grape supplier) and a demonstration that your drift report flags *exactly* those 2 features and none of the untouched 11 — the kind of regression test you'd keep in a monitoring test suite.

## 1. Setup and data

Nothing to implement yet — just load the real data, split it, and look at the class balance.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42

data = load_wine(as_frame=True)
X, y = data.data, data.target

print("shape:", X.shape)
print("class balance (0/1/2 = three cultivars):", y.value_counts().to_dict())
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

point_accuracy = accuracy_score(y_test, y_pred)
point_macro_f1 = f1_score(y_test, y_pred, average="macro")
print("point accuracy:", point_accuracy)
print("point macro-F1:", point_macro_f1)

## 2. Bootstrap confidence intervals for the reported metrics

The teammate's "98% accuracy" is a single number computed on one particular test split. Implement a **percentile bootstrap**: resample `(y_true, y_pred)` *pairs* with replacement many times, recompute the metric each time, and read off the 2.5th/97.5th percentiles of that distribution as a 95% CI.

Resample pairs together (not `y_true` and `y_pred` independently) — shuffling them separately would destroy the correspondence between a prediction and its true label.

TODO: implement `bootstrap_metric_ci`.

In [ ]:
def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    \"\"\"Percentile bootstrap CI for metric_fn(y_true, y_pred).

    Returns (lower, point_estimate, upper).
    \"\"\"
    # TODO:
    # 1. Convert y_true, y_pred to numpy arrays; let n = len(y_true).
    # 2. Compute point = metric_fn(y_true, y_pred) on the *original* (unresampled) data.
    # 3. For n_boot iterations: draw n indices with replacement (np.random.default_rng),
    #    index both y_true and y_pred with the SAME indices, and compute the metric.
    # 4. Return the (alpha/2, 1-alpha/2) percentiles of the bootstrap distribution as
    #    (lower, upper), alongside the original point estimate.
    raise NotImplementedError

In [ ]:
# Use bootstrap_metric_ci to get 95% CIs for accuracy and macro-F1 on the test set.
# acc_ci = bootstrap_metric_ci(...)
# f1_ci = bootstrap_metric_ci(..., metric_fn=lambda a, b: f1_score(a, b, average="macro"), ...)
# print("accuracy 95% CI:", acc_ci)
# print("macro-F1 95% CI:", f1_ci)

## 3. Simulating a production drift scenario

Real datasets don't come with a "drifted" copy attached, so to demonstrate a drift *detector* actually works, we need a batch where we **know** the ground truth of what shifted. We build it ourselves by taking the real test data and deliberately perturbing two of its columns — standing in for, e.g., a recalibrated alcohol sensor and a new color-measurement instrument. This injected shift is only used to validate the detector; the underlying 178 wine samples are real.

TODO: build a `production` DataFrame that is a copy of `X_test` with:
- `"alcohol"` shifted up by `+1.5` plus a little Gaussian noise (std 0.1)
- `"color_intensity"` scaled by `1.8x`
- every other column left untouched

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# TODO: production = X_test.copy()
# TODO: production["alcohol"] = ...
# TODO: production["color_intensity"] = ...
production = None

## 4. Per-feature drift detection with the Kolmogorov–Smirnov test

For each feature, run a two-sample KS test comparing the reference distribution (`X_test`) against the current distribution (`production`). The KS test compares two empirical distributions without assuming normality — appropriate here since we don't know the true distribution of these chemical measurements.

Running one hypothesis test per feature inflates the false-positive rate across the whole feature set, so apply a **Bonferroni correction**: divide `alpha` by the number of features before comparing it to each p-value.

TODO: implement `ks_drift_report`.

In [ ]:
def ks_drift_report(reference: pd.DataFrame, current: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    \"\"\"Per-column two-sample KS drift test, Bonferroni-corrected across columns.

    Returns a DataFrame with columns: feature, ks_stat, p_value, flagged.
    \"\"\"
    # TODO:
    # 1. bonferroni_alpha = alpha / reference.shape[1]
    # 2. For each column shared by reference and current, run scipy.stats.ks_2samp.
    # 3. flagged = p_value < bonferroni_alpha
    # 4. Return a DataFrame of the rows, sorted by p_value ascending.
    raise NotImplementedError

In [ ]:
# report = ks_drift_report(X_test, production)
# report

# Regression-test-style assertions for your drift suite:
# flagged = set(report.loc[report["flagged"], "feature"])
# assert {"alcohol", "color_intensity"} <= flagged
# assert len(flagged - {"alcohol", "color_intensity"}) == 0

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    rng = np.random.default_rng(random_state)

    point = metric_fn(y_true, y_pred)
    boot_scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_scores[i] = metric_fn(y_true[idx], y_pred[idx])

    lower = np.percentile(boot_scores, 100 * alpha / 2)
    upper = np.percentile(boot_scores, 100 * (1 - alpha / 2))
    return lower, point, upper


acc_ci = bootstrap_metric_ci(y_test.to_numpy(), y_pred, accuracy_score, random_state=RANDOM_STATE)
f1_ci = bootstrap_metric_ci(
    y_test.to_numpy(), y_pred,
    lambda a, b: f1_score(a, b, average="macro"),
    random_state=RANDOM_STATE,
)
print("accuracy 95% CI:", acc_ci)   # e.g. (0.944, 0.981, 1.000)
print("macro-F1 95% CI:", f1_ci)    # e.g. (0.943, 0.983, 1.000)


rng = np.random.default_rng(RANDOM_STATE)
production = X_test.copy()
production["alcohol"] = production["alcohol"] + 1.5 + rng.normal(0, 0.1, size=len(production))
production["color_intensity"] = production["color_intensity"] * 1.8


def ks_drift_report(reference, current, alpha=0.05):
    bonferroni_alpha = alpha / reference.shape[1]
    rows = []
    for col in reference.columns:
        stat, p_value = ks_2samp(reference[col], current[col])
        rows.append({
            "feature": col,
            "ks_stat": stat,
            "p_value": p_value,
            "flagged": p_value < bonferroni_alpha,
        })
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)


report = ks_drift_report(X_test, production)
print(report)

flagged = set(report.loc[report["flagged"], "feature"])
assert {"alcohol", "color_intensity"} <= flagged
assert len(flagged - {"alcohol", "color_intensity"}) == 0
print("Drift suite correctly isolated the 2 shifted features out of 13.")
```

**Why this matters for ML quality work:**

- A **point estimate** of a metric is a single draw from a sampling distribution. On a ~54-row test set, the 95% CI for accuracy here is roughly 9 points wide (e.g. 94%–100%) — "98% accuracy" and "95% accuracy" could easily be the *same underlying model* evaluated on different random splits. Any go/no-go decision, or any claim that model B beat model A, should be backed by an interval or a proper paired significance test, not a single number.
- Resampling `(y_true, y_pred)` pairs (rather than resampling each independently) preserves the per-example correctness pattern, which is what makes this a valid bootstrap for a *classification metric* rather than nonsense.
- The **Bonferroni correction** matters once you're running a KS test per feature: with 13 features and an uncorrected alpha of 0.05, you'd expect roughly 0.65 false drift alarms per batch on pure noise alone, even with zero real drift. Dividing alpha by the number of tests keeps the whole-suite false-positive rate under control — exactly the same logic as adjusting flakiness thresholds across a large regression suite.
- The reason this exercise manufactures the drift by perturbing two real columns, rather than looking for a dataset that happens to already contain labeled drift, is so the "ground truth" (which features actually shifted) is known — this is what lets you write an `assert` that this specific test suite is doing its job, rather than just eyeballing a p-value table.

</details>